# Survival analysis with C-MAPSS

Survival analysis is a time-to-event method that represents both event times
and right-censored observations. This example combines failed training engines
with test engines censored at their final observed cycle.

## Learning goals

- construct one duration and event indicator per engine;
- understand right censoring;
- compute a Kaplan-Meier estimate without a specialized survival package.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata

In [ ]:
pdmdata.download("cmapss")
train = pdmdata.load("cmapss", subset="FD001", split="train")
test = pdmdata.load("cmapss", subset="FD001", split="test")

failed = (
    train.group_by("unit_number")
    .agg(pl.col("cycle").max().alias("duration"))
    .with_columns(
        pl.lit(True).alias("event_observed"),
        pl.lit("train failure").alias("cohort"),
    )
)
censored = (
    test.group_by("unit_number")
    .agg(pl.col("cycle").max().alias("duration"))
    .with_columns(
        pl.lit(False).alias("event_observed"),
        pl.lit("test cutoff").alias("cohort"),
    )
)
survival_data = pl.concat([failed, censored])
survival_data.group_by("cohort").agg(
    pl.len().alias("engines"),
    pl.col("duration").median().alias("median_observed_cycles"),
)

## Kaplan-Meier estimator

At each failure time, the conditional survival probability is
`1 - failures / engines_at_risk`. Censored engines reduce later risk sets but
do not create a survival drop.

In [ ]:
event_table = (
    survival_data.group_by("duration")
    .agg(
        pl.col("event_observed").sum().alias("events"),
        (~pl.col("event_observed")).sum().alias("censored"),
    )
    .sort("duration")
)

durations = event_table["duration"].to_numpy()
events = event_table["events"].to_numpy()
at_risk = np.array([
    survival_data.filter(pl.col("duration") >= duration).height
    for duration in durations
])
conditional = np.where(events > 0, 1 - events / at_risk, 1.0)
estimate = event_table.with_columns(
    pl.Series("at_risk", at_risk),
    pl.Series("survival_probability", np.cumprod(conditional)),
)
estimate.head()

In [ ]:
plt.step(
    estimate["duration"].to_numpy(),
    estimate["survival_probability"].to_numpy(),
    where="post",
)
plt.ylim(0, 1.02)
plt.xlabel("Cycles since first observation")
plt.ylabel("Estimated survival probability")
plt.show()

## Interpretation and limitations

This mixed train/test cohort is instructional, not an unbiased population:
C-MAPSS test cutoffs were constructed for a benchmark. Covariate-aware survival
models also require careful landmarking to avoid treating future sensor values
as baseline information.